---

# 스프린트미션10 4팀_김명환

---
---

---
---

# >프로그램< 기본 라이브러리 로드

In [47]:
from urllib.request import urlretrieve
urlretrieve("https://raw.githubusercontent.com/c0z0c/jupyter_hangul/refs/heads/beta/helper_utils.py", "helper_utils.py")
import importlib
import helper_utils as hu
importlib.reload(hu)
from helper_utils import *

# 표준/유틸
from pathlib import Path
import logging
import random
import math
import os
import sys
import json
import yaml
import zipfile
import shutil
import re
import unicodedata
import html
import pickle
from collections import OrderedDict, Counter, defaultdict
from typing import Dict, List, Tuple
from abc import ABC, abstractmethod

# 데이터/시각화
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import xml.etree.ElementTree as ET
from datetime import datetime, timezone, timedelta
import pytz

# Scikit-learn (필요 모듈만 그룹화)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.datasets import (
    fetch_california_housing, load_iris, make_moons, make_circles,
    load_breast_cancer, load_wine
)
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, mean_squared_error, average_precision_score

# 이미지 관련
from PIL import Image, ImageFilter, ImageDraw
import albumentations as A
import IPython.display as display
from tqdm.notebook import tqdm

# PyTorch 관련
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader, TensorDataset, RandomSampler
from torchvision.transforms import v2
from torchvision.datasets import CocoDetection
from torchvision.transforms import functional as TF
from torch.nn import CrossEntropyLoss

# --- 전역 설정 (규약 준수: 변수명 유지) ---
__kst = pytz.timezone('Asia/Seoul')
__device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
__device_cpu = torch.device('cpu')

# 재현성 및 시드 고정
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if __device.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)
    # 결정론적 설정 (필요 시 해제 가능)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# 로깅 설정 (노트북에서는 INFO로 충분)
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logging.info(f"라이브러리 로드 완료. 사용장치: {__device}")

2025-10-10 23:05:07,680 INFO 라이브러리 로드 완료. 사용장치: cpu


🌐 https://c0z0c.github.io/jupyter_hangul
✅ 설정 완료: 한글 폰트, plt 전역 등록, pandas 확장, 캐시 기능
pd commit 저장 경로 = d:\GoogleDrive\homepage\스프린트미션\스프린트미션_작업중


In [48]:
def get_path_temp(add_path = None):
    if IS_COLAB:
        temp_path = r"/content/temp"
    else:
        drive = os.path.splitdrive(os.getcwd())[0]  # ex: 'D:'
        temp_path = os.path.join(drive + os.sep, 'temp')
    if add_path is not None:
        temp_path = os.path.join(temp_path,add_path)
    return temp_path

drive_temp_path = get_path_temp()
os.makedirs(drive_temp_path, exist_ok=True)

drive_temp_download_path = os.path.join(drive_temp_path, r'일상생활및_구체어_clean_data.zip')

print(f"temp path: {drive_temp_path}")
print(f"temp download path: {drive_temp_download_path}")


temp path: d:\temp
temp download path: d:\temp\일상생활및_구체어_clean_data.zip


# >프로그램< 데이터 전처리

In [49]:
# https://drive.google.com/file/d/1BJha3iuE9oB3Tdq1XCGuZDMUm6HLlh48/view?usp=sharing
import gdown
if not os.path.exists(drive_temp_download_path):
    url = "https://drive.google.com/uc?id=1BJha3iuE9oB3Tdq1XCGuZDMUm6HLlh48"
    gdown.download(url, drive_temp_download_path, quiet=False)
# 다운로드 파일 사이즈
print(f"다운로드 파일 : {drive_temp_download_path} {os.path.getsize(drive_temp_download_path)} bytes")

다운로드 파일 : d:\temp\일상생활및_구체어_clean_data.zip 65726573 bytes


In [50]:
# ===========================
# 2. BaseTokenizer (추상 클래스)
# ===========================
class BaseTokenizer(ABC):
    """모든 토크나이저의 공통 인터페이스"""
    
    @abstractmethod
    def tokenize(self, text: str) -> List[str]:
        """텍스트를 토큰 리스트로 변환"""
        pass
    
    @abstractmethod
    def detokenize(self, tokens: List[str]) -> str:
        """토큰 리스트를 텍스트로 복원"""
        pass
    
    @abstractmethod
    def get_tokenizer_name(self) -> str:
        """토크나이저 이름 반환"""
        pass


# ===========================
# 3. JamoTokenizer (자소 분해)
# ===========================
class JamoTokenizer(BaseTokenizer):
    """한글 자소 단위 토크나이저"""
    
    def __init__(self):
        # 초성 19개
        self.CHO = ['ㄱ', 'ㄲ', 'ㄴ', 'ㄷ', 'ㄸ', 'ㄹ', 'ㅁ', 'ㅂ', 'ㅃ', 
                    'ㅅ', 'ㅆ', 'ㅇ', 'ㅈ', 'ㅉ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ']
        # 중성 21개
        self.JUNG = ['ㅏ', 'ㅐ', 'ㅑ', 'ㅒ', 'ㅓ', 'ㅔ', 'ㅕ', 'ㅖ', 'ㅗ', 'ㅘ',
                     'ㅙ', 'ㅚ', 'ㅛ', 'ㅜ', 'ㅝ', 'ㅞ', 'ㅟ', 'ㅠ', 'ㅡ', 'ㅢ', 'ㅣ']
        # 종성 28개 (빈 종성 포함)
        self.JONG = ['', 'ㄱ', 'ㄲ', 'ㄳ', 'ㄴ', 'ㄵ', 'ㄶ', 'ㄷ', 'ㄹ', 'ㄺ',
                     'ㄻ', 'ㄼ', 'ㄽ', 'ㄾ', 'ㄿ', 'ㅀ', 'ㅁ', 'ㅂ', 'ㅄ', 'ㅅ',
                     'ㅆ', 'ㅇ', 'ㅈ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ']
        
        self.HANGEUL_BASE = 0xAC00
        self.HANGEUL_END = 0xD7A3
    
    def _is_hangeul(self, char: str) -> bool:
        """한글 음절인지 확인"""
        if len(char) != 1:
            return False
        code = ord(char)
        return self.HANGEUL_BASE <= code <= self.HANGEUL_END
    
    def _decompose_char(self, char: str) -> List[str]:
        """한글 1글자를 자소로 분해"""
        if not self._is_hangeul(char):
            return [char]  # 한글이 아니면 그대로 반환
        
        code = ord(char) - self.HANGEUL_BASE
        jong_idx = code % 28
        jung_idx = ((code - jong_idx) // 28) % 21
        cho_idx = ((code - jong_idx) // 28) // 21
        
        jamos = [self.CHO[cho_idx], self.JUNG[jung_idx]]
        if jong_idx > 0:
            jamos.append(self.JONG[jong_idx])
        
        return jamos
    
    def _compose_jamos(self, jamos: List[str]) -> str:
        """자소 리스트를 한글 음절로 복원"""
        if len(jamos) < 2:
            return ''.join(jamos)
        
        cho = jamos[0]
        jung = jamos[1]
        jong = jamos[2] if len(jamos) > 2 else ''
        
        try:
            cho_idx = self.CHO.index(cho)
            jung_idx = self.JUNG.index(jung)
            jong_idx = self.JONG.index(jong)
            
            code = self.HANGEUL_BASE + (cho_idx * 21 + jung_idx) * 28 + jong_idx
            return chr(code)
        except (ValueError, IndexError):
            return ''.join(jamos)
    
    def tokenize(self, text: str) -> List[str]:
        """텍스트를 자소 단위로 분해"""
        tokens = []
        for char in text:
            if self._is_hangeul(char):
                tokens.extend(self._decompose_char(char))
            else:
                tokens.append(char)  # 영어, 숫자, 공백, 구두점 등
        return tokens
    
    def detokenize(self, tokens: List[str]) -> str:
        """자소 토큰을 텍스트로 복원"""
        result = []
        i = 0
        
        while i < len(tokens):
            # 3개 자소 시도 (초+중+종)
            if i + 2 < len(tokens):
                char = self._compose_jamos(tokens[i:i+3])
                if self._is_hangeul(char):
                    result.append(char)
                    i += 3
                    continue
            
            # 2개 자소 시도 (초+중)
            if i + 1 < len(tokens):
                char = self._compose_jamos(tokens[i:i+2])
                if self._is_hangeul(char):
                    result.append(char)
                    i += 2
                    continue
            
            # 자소 조합 실패 시 그대로 추가
            result.append(tokens[i])
            i += 1
        
        return ''.join(result)
    
    def get_tokenizer_name(self) -> str:
        return "jamo"


# ===========================
# 4. SyllableTokenizer (음절)
# ===========================
class SyllableTokenizer(BaseTokenizer):
    """음절 단위 토크나이저"""
    
    def tokenize(self, text: str) -> List[str]:
        """텍스트를 음절(문자) 단위로 분리"""
        return list(text)
    
    def detokenize(self, tokens: List[str]) -> str:
        """음절 토큰을 텍스트로 결합"""
        return ''.join(tokens)
    
    def get_tokenizer_name(self) -> str:
        return "syllable"


# ===========================
# 5. MorphemeTokenizer (형태소)
# ===========================
class MorphemeTokenizer(BaseTokenizer):
    """형태소 단위 토크나이저 (Mecab 사용)"""
    
    def __init__(self, use_mecab=True):
        self.use_mecab = use_mecab
        self.mecab = None
        
        if use_mecab:
            try:
                import MeCab
                self.mecab = MeCab.Tagger()
            except ImportError:
                print("Warning: MeCab not installed. Using syllable tokenization.")
                self.use_mecab = False
    
    def tokenize(self, text: str) -> List[str]:
        """텍스트를 형태소 단위로 분리"""
        if self.use_mecab and self.mecab:
            parsed = self.mecab.parse(text)
            tokens = []
            for line in parsed.split('\n'):
                if '\t' in line:
                    token = line.split('\t')[0]
                    tokens.append(token)
            return tokens
        else:
            # Mecab 없으면 음절 단위로 fallback
            return list(text)
    
    def detokenize(self, tokens: List[str]) -> str:
        """형태소 토큰을 텍스트로 결합 (완벽한 복원 어려움)"""
        return ''.join(tokens)
    
    def get_tokenizer_name(self) -> str:
        return "morpheme"


# ===========================
# 6. BaseVocabulary (추상 클래스)
# ===========================
class BaseVocabulary(ABC):
    """어휘 사전 공통 인터페이스"""
    
    def __init__(self):
        self.token2idx: Dict[str, int] = {}
        self.idx2token: Dict[int, str] = {}
        self.PAD_IDX = 0
        self.UNK_IDX = 1
        self.SOS_IDX = 2
        self.EOS_IDX = 3
        
        # 특수 토큰 초기화
        self._init_special_tokens()
    
    def _init_special_tokens(self):
        """특수 토큰 초기화"""
        special_tokens = ['<PAD>', '<UNK>', '<SOS>', '<EOS>']
        for idx, token in enumerate(special_tokens):
            self.token2idx[token] = idx
            self.idx2token[idx] = token
    
    @abstractmethod
    def build_vocab(self, tokens_list: List[List[str]], min_freq: int = 1):
        """토큰 리스트에서 어휘 사전 구축"""
        pass
    
    def encode(self, tokens: List[str]) -> List[int]:
        """토큰을 인덱스로 변환"""
        return [self.token2idx.get(token, self.UNK_IDX) for token in tokens]
    
    def decode(self, indices: List[int]) -> List[str]:
        """인덱스를 토큰으로 변환"""
        return [self.idx2token.get(idx, '<UNK>') for idx in indices]
    
    def __len__(self) -> int:
        """어휘 크기"""
        return len(self.token2idx)
    
    def save(self, path: str):
        """어휘 사전 저장"""
        with open(path, 'w', encoding='utf-8') as f:
            json.dump({
                'token2idx': self.token2idx,
                'idx2token': {int(k): v for k, v in self.idx2token.items()}
            }, f, ensure_ascii=False, indent=2)
    
    def load(self, path: str):
        """어휘 사전 로드"""
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            self.token2idx = data['token2idx']
            self.idx2token = {int(k): v for k, v in data['idx2token'].items()}


# ===========================
# 7. JamoVocabulary (자소 어휘)
# ===========================
class JamoVocabulary(BaseVocabulary):
    """자소 기반 고정 어휘 사전"""
    
    def __init__(self):
        super().__init__()
        self._build_fixed_vocab()
    
    def _build_fixed_vocab(self):
        """고정 자소 어휘 구축"""
        # 초성
        cho = ['ㄱ', 'ㄲ', 'ㄴ', 'ㄷ', 'ㄸ', 'ㄹ', 'ㅁ', 'ㅂ', 'ㅃ', 
               'ㅅ', 'ㅆ', 'ㅇ', 'ㅈ', 'ㅉ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ']
        # 중성
        jung = ['ㅏ', 'ㅐ', 'ㅑ', 'ㅒ', 'ㅓ', 'ㅔ', 'ㅕ', 'ㅖ', 'ㅗ', 'ㅘ',
                'ㅙ', 'ㅚ', 'ㅛ', 'ㅜ', 'ㅝ', 'ㅞ', 'ㅟ', 'ㅠ', 'ㅡ', 'ㅢ', 'ㅣ']
        # 종성
        jong = ['ㄱ', 'ㄲ', 'ㄳ', 'ㄴ', 'ㄵ', 'ㄶ', 'ㄷ', 'ㄹ', 'ㄺ',
                'ㄻ', 'ㄼ', 'ㄽ', 'ㄾ', 'ㄿ', 'ㅀ', 'ㅁ', 'ㅂ', 'ㅄ', 'ㅅ',
                'ㅆ', 'ㅇ', 'ㅈ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ']
        
        # 영어 소문자, 대문자
        alphabet = [chr(i) for i in range(ord('a'), ord('z')+1)]
        alphabet += [chr(i) for i in range(ord('A'), ord('Z')+1)]
        
        # 숫자
        digits = [str(i) for i in range(10)]
        
        # 구두점 및 기호
        punctuation = [' ', '.', ',', '!', '?', ';', ':', '-', '(', ')', 
                      '"', "'", '/', '&', '@', '#', '$', '%']
        
        # 모든 토큰 합치기
        all_tokens = cho + jung + jong + alphabet + digits + punctuation
        
        # 인덱스 할당 (특수 토큰 이후부터)
        current_idx = 4  # PAD, UNK, SOS, EOS 이후
        for token in all_tokens:
            if token not in self.token2idx:
                self.token2idx[token] = current_idx
                self.idx2token[current_idx] = token
                current_idx += 1
    
    def build_vocab(self, tokens_list: List[List[str]], min_freq: int = 1):
        """자소는 고정 어휘이므로 build_vocab 불필요"""
        pass  # 이미 초기화 시 구축됨


# ===========================
# 8. DynamicVocabulary (동적 어휘)
# ===========================
class DynamicVocabulary(BaseVocabulary):
    """음절/형태소용 동적 어휘 사전"""
    
    def build_vocab(self, tokens_list: List[List[str]], min_freq: int = 1):
        """데이터 기반 어휘 사전 구축"""
        # 토큰 빈도 계산
        counter = Counter()
        for tokens in tokens_list:
            counter.update(tokens)
        
        # 빈도 필터링 및 인덱스 할당
        current_idx = 4  # 특수 토큰 이후
        for token, freq in counter.items():
            if freq >= min_freq and token not in self.token2idx:
                self.token2idx[token] = current_idx
                self.idx2token[current_idx] = token
                current_idx += 1
        
        print(f"Vocabulary size: {len(self.token2idx)}")


# ===========================
# 9. DataPipeline (통합 파이프라인)
# ===========================
class DataPipeline:
    """전체 전처리 파이프라인"""
    
    def __init__(self, tokenizer_type='jamo'):
        """
        Args:
            tokenizer_type: 'jamo', 'syllable', 'morpheme'
        """
        self.tokenizer_type = tokenizer_type
        
        # 토크나이저 선택
        if tokenizer_type == 'jamo':
            self.ko_tokenizer = JamoTokenizer()
            self.mt_tokenizer = JamoTokenizer()
        elif tokenizer_type == 'syllable':
            self.ko_tokenizer = SyllableTokenizer()
            self.mt_tokenizer = SyllableTokenizer()
        elif tokenizer_type == 'morpheme':
            self.ko_tokenizer = MorphemeTokenizer()
            self.mt_tokenizer = SyllableTokenizer()  # 영어는 음절
        else:
            raise ValueError(f"Unknown tokenizer type: {tokenizer_type}")
        
        print(f"Initialized with {tokenizer_type} tokenizer")
    
    def load_data(self, json_path: str) -> List[Dict]:
        """JSON 파일 로드"""
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"Loaded {len(data)} samples from {json_path}")
        return data
    
    def tokenize_dataset(self, data: List[Dict]) -> List[Dict]:
        """전체 데이터셋 토큰화"""
        tokenized = []
        for sample in tqdm(data, desc="Tokenizing"):
            ko_tokens = self.ko_tokenizer.tokenize(sample['ko'])
            mt_tokens = self.mt_tokenizer.tokenize(sample['mt'])
            
            tokenized.append({
                'ko_tokens': ko_tokens,
                'mt_tokens': mt_tokens
            })
        
        return tokenized
    
    def build_vocabularies(self, tokenized_data: List[Dict], min_freq: int = 1):
        """어휘 사전 구축"""
        # 어휘 클래스 선택
        if self.tokenizer_type == 'jamo':
            ko_vocab = JamoVocabulary()
            mt_vocab = JamoVocabulary()
        else:
            ko_vocab = DynamicVocabulary()
            mt_vocab = DynamicVocabulary()
            
            # 동적 어휘는 build 필요
            ko_tokens_list = [s['ko_tokens'] for s in tokenized_data]
            mt_tokens_list = [s['mt_tokens'] for s in tokenized_data]
            
            print("Building Korean vocabulary...")
            ko_vocab.build_vocab(ko_tokens_list, min_freq)
            
            print("Building English vocabulary...")
            mt_vocab.build_vocab(mt_tokens_list, min_freq)
        
        return ko_vocab, mt_vocab
    
    def pad_sequence(self, tokens: List[str], vocab: BaseVocabulary, 
                     max_length: int, add_eos: bool = True) -> List[int]:
        """토큰을 인덱스로 변환하고 패딩"""
        indices = vocab.encode(tokens)
        
        if add_eos:
            indices.append(vocab.EOS_IDX)
        
        # 트렁케이션
        if len(indices) > max_length:
            indices = indices[:max_length-1] + [vocab.EOS_IDX]
        else:
            indices += [vocab.PAD_IDX] * (max_length - len(indices))
        
        return indices
    
    def encode_dataset(self, tokenized_data: List[Dict], 
                      ko_vocab: BaseVocabulary, 
                      mt_vocab: BaseVocabulary,
                      max_length: int = 200) -> List[Tuple]:
        """토큰을 인덱스로 인코딩"""
        encoded = []
        
        for sample in tqdm(tokenized_data, desc="Encoding"):
            # Source (한국어)
            src = self.pad_sequence(sample['ko_tokens'], ko_vocab, max_length)
            
            # Target input (Teacher Forcing용)
            tgt_tokens = sample['mt_tokens']
            tgt_input = [mt_vocab.SOS_IDX] + mt_vocab.encode(tgt_tokens)
            
            if len(tgt_input) > max_length:
                tgt_input = tgt_input[:max_length]
            else:
                tgt_input += [mt_vocab.PAD_IDX] * (max_length - len(tgt_input))
            
            # Target output
            tgt_output = mt_vocab.encode(tgt_tokens) + [mt_vocab.EOS_IDX]
            
            if len(tgt_output) > max_length:
                tgt_output = tgt_output[:max_length-1] + [mt_vocab.EOS_IDX]
            else:
                tgt_output += [mt_vocab.PAD_IDX] * (max_length - len(tgt_output))
            
            encoded.append((src, tgt_input, tgt_output))
        
        return encoded
    
    def create_dataloader(self, encoded_data: List[Tuple], 
                          batch_size: int = 128,
                          num_workers: int = 2,
                          shuffle: bool = False
                          ) -> DataLoader:
        """TensorDataset 및 DataLoader 생성"""
        # 텐서 데이터
        data_src = torch.tensor([x[0] for x in encoded_data], dtype=torch.long)
        data_tgt_in = torch.tensor([x[1] for x in encoded_data], dtype=torch.long)
        data_tgt_out = torch.tensor([x[2] for x in encoded_data], dtype=torch.long)
        tensor_dataset = TensorDataset(data_src, data_tgt_in, data_tgt_out)
        print(f"Loader dataset size: {len(tensor_dataset)}")
        
        # DataLoader
        loader = DataLoader(
            tensor_dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available()
        )
        print(f"Loader batches: {len(loader)}")

        return loader


In [51]:
def extract_dataset_from_zip(zip_path: str) -> str:
    """
    ZIP 파일 압축 해제 및 데이터 디렉토리 반환
    
    Args:
        zip_path: ZIP 파일 경로
    
    Returns:
        압축 해제된 데이터 디렉토리 경로
    """
    extract_dir = zip_path + ".unzip"
    
    # 이미 압축 해제된 경우 스킵
    if os.path.exists(extract_dir) and os.listdir(extract_dir):
        logging.info(f"압축 해제됨 {extract_dir}")
        return extract_dir
    
    # helper_utils.unzip() 사용
    logging.info(f"압축 해제 {zip_path} to {extract_dir}...")
    extracted_paths = unzip([zip_path], remove_zip=False)
    
    if not extracted_paths:
        raise RuntimeError(f"Failed to extract {zip_path}")
    
    # unzip()은 압축 해제 후 디렉토리 리스트 반환
    actual_extract_dir = extracted_paths[0]

    logging.info(f"압축 해제 완료. 파일: {os.listdir(actual_extract_dir)}")
    return actual_extract_dir

# >프로그램< 로직 테스트

In [52]:

def create_loader(drive_temp_download_path: str, test_size: int = 1000):
    """
    토큰화 파이프라인 실행 및 DataLoader 생성
    
    Args:
        drive_temp_download_path: 다운로드된 ZIP 파일 경로
    """
    # 1. ZIP 압축 해제
    data_dir = extract_dataset_from_zip(drive_temp_download_path)
    print(f"Step 1: Data directory: {data_dir}")
    
    # 2. 파이프라인 생성
    pipeline = DataPipeline(tokenizer_type='jamo')
    print(f"Step 2: Pipeline created with tokenizer {pipeline.tokenizer_type}")
    
    # 3. 데이터 로드 (동적 경로)
    train_json_path = os.path.join(data_dir, 'clean_train.json')
    valid_json_path = os.path.join(data_dir, 'clean_valid.json')
    print(f"Step 3: {train_json_path} {os.path.exists(train_json_path)} {valid_json_path} {os.path.exists(valid_json_path)}")
    
    train_data = pipeline.load_data(train_json_path)
    valid_data = pipeline.load_data(valid_json_path)
    
    # 4. 토큰화
    tokenized_train = pipeline.tokenize_dataset(train_data)
    tokenized_valid = pipeline.tokenize_dataset(valid_data)
    # tokenized_train 데이터 일부만 사용 (최대 1% 또는 test_size)
    tokenized_train, tokenized_test = train_test_split(
        tokenized_train,
        test_size=min(test_size, len(tokenized_train) // 100),  # 최대 1%
        random_state=SEED,
        shuffle=True
    )

    print(f'Step 4: 토큰 후 사이즈 {len(tokenized_train)}, {len(tokenized_valid)} {len(tokenized_test)}')
    
    # 5. 어휘 사전 구축
    ko_vocab, mt_vocab = pipeline.build_vocabularies(tokenized_train, min_freq=2)
    print(f'Step 5: 어휘 사전 크기 {len(ko_vocab)}, {len(mt_vocab)}')
    
    # 6. 인코딩
    encoded_train = pipeline.encode_dataset(tokenized_train, ko_vocab, mt_vocab, max_length=200)
    encoded_valid = pipeline.encode_dataset(tokenized_valid, ko_vocab, mt_vocab, max_length=200)
    encoded_test = pipeline.encode_dataset(tokenized_test, ko_vocab, mt_vocab, max_length=200)
    print(f'Step 6: 인코딩 후 사이즈 {len(encoded_train)}, {len(encoded_valid)}, {len(encoded_test)}')

    # 7. DataLoader 생성
    train_loader = pipeline.create_dataloader(encoded_train, batch_size=128, num_workers=1, shuffle=True)
    valid_loader = pipeline.create_dataloader(encoded_valid, batch_size=128, num_workers=1, shuffle=False)
    test_loader = pipeline.create_dataloader(encoded_test, batch_size=128, num_workers=1, shuffle=False)
    print(f'Step 7: DataLoader 생성 완료')
    
    # 8. 어휘 사전 저장 (동적 경로)
    vocab_dir = get_path_temp('vocab')
    os.makedirs(vocab_dir, exist_ok=True)
    
    ko_vocab_path = os.path.join(vocab_dir, 'ko_vocab.json')
    mt_vocab_path = os.path.join(vocab_dir, 'mt_vocab.json')
    
    ko_vocab.save(ko_vocab_path)
    mt_vocab.save(mt_vocab_path)
    
    print(f"Step 8: Vocabularies saved to {vocab_dir}")
    
    # 9. 샘플 출력
    for batch in train_loader:
        src, tgt_in, tgt_out = batch
        print(f"Source shape: {src.shape}")
        print(f"Target input shape: {tgt_in.shape}")
        print(f"Target output shape: {tgt_out.shape}")
        break
    
    print("Step 9: Tokenization pipeline ready!")
    return train_loader, valid_loader, test_loader, ko_vocab, mt_vocab

# 실행
train_loader, valid_loader, test_loader, ko_vocab, mt_vocab = create_loader(drive_temp_download_path)


2025-10-10 23:05:07,917 INFO 압축 해제됨 d:\temp\일상생활및_구체어_clean_data.zip.unzip


Step 1: Data directory: d:\temp\일상생활및_구체어_clean_data.zip.unzip
Initialized with jamo tokenizer
Step 2: Pipeline created with tokenizer jamo
Step 3: d:\temp\일상생활및_구체어_clean_data.zip.unzip\clean_train.json True d:\temp\일상생활및_구체어_clean_data.zip.unzip\clean_valid.json True
Loaded 1031323 samples from d:\temp\일상생활및_구체어_clean_data.zip.unzip\clean_train.json
Loaded 136175 samples from d:\temp\일상생활및_구체어_clean_data.zip.unzip\clean_valid.json


Tokenizing:   0%|          | 0/1031323 [00:00<?, ?it/s]

Tokenizing:   0%|          | 0/136175 [00:00<?, ?it/s]

Step 4: 토큰 후 사이즈 1030323, 136175 1000
Step 5: 어휘 사전 크기 135, 135


Encoding:   0%|          | 0/1030323 [00:00<?, ?it/s]

Encoding:   0%|          | 0/136175 [00:00<?, ?it/s]

Encoding:   0%|          | 0/1000 [00:00<?, ?it/s]

Step 6: 인코딩 후 사이즈 1030323, 136175, 1000
Loader dataset size: 1030323
Loader batches: 8050
Loader dataset size: 136175
Loader batches: 1064
Loader dataset size: 1000
Loader batches: 8
Step 7: DataLoader 생성 완료
Step 8: Vocabularies saved to d:\temp\vocab
Source shape: torch.Size([128, 200])
Target input shape: torch.Size([128, 200])
Target output shape: torch.Size([128, 200])
Step 9: Tokenization pipeline ready!


In [53]:
print(f"Train batches: {len(train_loader)}, Valid batches: {len(valid_loader)}, Test batches: {len(test_loader)}")
print(f"Korean Vocab size: {len(ko_vocab)}, English Vocab size: {len(mt_vocab)}")

Train batches: 8050, Valid batches: 1064, Test batches: 8
Korean Vocab size: 135, English Vocab size: 135
